## Finding Dataset

In order to realize the study, datasets representing the crimes were searched for, with the City of Chicago being considered the best. It offered an easy api to obtain data as a detailed dataset displaying crimes per day, which was of great value for the project. The "$limit=5000000" filter was added in order to be able to download all entries, as the API seems to have a default limit a little over a thousand, and the crude dataset goes beyond a million.

Before continuing, the dataset was downloaded to a file for the manual read of the data.

In [2]:
import pandas as pd

url = "https://data.cityofchicago.org/resource/ijzp-q8t2.csv?$where=date>'2020-01-01'&$limit=5000000"

df = pd.read_csv(url)

df.head()

print("Downloaded Chicago crimes datasets over 2020", len(df), "records")
print(df.columns)


Downloaded Chicago crimes datasets over 2020 1432194 records
Index(['id', 'case_number', 'date', 'block', 'iucr', 'primary_type',
       'description', 'location_description', 'arrest', 'domestic', 'beat',
       'district', 'ward', 'community_area', 'fbi_code', 'x_coordinate',
       'y_coordinate', 'year', 'updated_on', 'latitude', 'longitude',
       'location'],
      dtype='object')


After the retrieval and storage of the raw dataset, the step of cleaning and deciding what columns to drop and which ones to keep, is started

In [13]:
df.columns

Index(['id', 'case_number', 'date', 'block', 'iucr', 'primary_type',
       'description', 'location_description', 'arrest', 'domestic', 'beat',
       'district', 'ward', 'community_area', 'fbi_code', 'x_coordinate',
       'y_coordinate', 'year', 'updated_on', 'latitude', 'longitude',
       'location', 'date_only'],
      dtype='object')

Out of the columns of the dataset, the ones considered important were: 

- date
- primary_type
- location_description
- arrest
- domestic

Out of these, the column date was grouped by days, as it is the same time scale used for the other datasets.

In [5]:
df["date"] = pd.to_datetime(df["date"], errors="coerce")

df["date_only"] = df["date"].dt.date

crime_stats_clean = (
    df
    .groupby(["date_only", "primary_type","location_description", "arrest", "domestic"])
    .size()
    .reset_index(name="crime_count")
)

crime_stats_clean.head()

crime_stats_clean.to_csv(
    "crime_stats_clean.csv",
    index=False
)

print(len(crime_stats_clean), "rows in grouped dataset\n\n")

print(df["primary_type"].unique(), "\n\n")

print(df["location_description"].unique(), "\n\n")

print(df["arrest"].unique())

print(df["domestic"].unique())

485251 rows in grouped dataset


['BATTERY' 'THEFT' 'MOTOR VEHICLE THEFT' 'CRIMINAL DAMAGE' 'ASSAULT'
 'OFFENSE INVOLVING CHILDREN' 'OTHER OFFENSE' 'ROBBERY'
 'WEAPONS VIOLATION' 'INTERFERENCE WITH PUBLIC OFFICER' 'NARCOTICS'
 'SEX OFFENSE' 'PROSTITUTION' 'CRIMINAL TRESPASS' 'BURGLARY'
 'DECEPTIVE PRACTICE' 'PUBLIC PEACE VIOLATION' 'CRIMINAL SEXUAL ASSAULT'
 'LIQUOR LAW VIOLATION' 'ARSON' 'INTIMIDATION' 'STALKING' 'HOMICIDE'
 'CONCEALED CARRY LICENSE VIOLATION' 'HUMAN TRAFFICKING'
 'PUBLIC INDECENCY' 'KIDNAPPING' 'OBSCENITY' 'GAMBLING'
 'OTHER NARCOTIC VIOLATION' 'NON-CRIMINAL' 'RITUALISM'
 'CRIM SEXUAL ASSAULT'] 


['RESIDENCE' 'VEHICLE NON-COMMERCIAL' 'STREET' 'SIDEWALK' 'APARTMENT'
 'ALLEY' 'RESIDENCE - YARD (FRONT / BACK)' 'BAR OR TAVERN'
 'AIRPORT PARKING LOT' 'GAS STATION' 'OTHER (SPECIFY)'
 'PARKING LOT / GARAGE (NON RESIDENTIAL)' 'CONVENIENCE STORE'
 'COMMERCIAL / BUSINESS OFFICE' 'RESTAURANT' 'SMALL RETAIL STORE'
 'DEPARTMENT STORE' 'SCHOOL - PRIVATE GROUNDS'
 'HOSPITAL BUILDI

After the grouping, the cleaning of the dataset takes place. The first clean function ```clean_primary_type(df)``` focus on the primary type column. The logic consists of ensuring that all data is the same data format, any data format discrepancy is removed and values are normalized.

In [6]:
def clean_primary_type(df):
    """
    Fully cleans and standardizes the 'primary_type' column.
    Handles case variation, spacing, punctuation, and merges inconsistent labels.
    """

    df["primary_type"] = (
        df["primary_type"]
        .astype(str)
        .str.upper()
        .str.strip()
    )

    df["primary_type"] = (
        df["primary_type"]
        .str.replace(r"[^A-Z0-9\s]", "", regex=True)   
        .str.replace(r"\s+", " ", regex=True)          
        .str.strip()
    )

    df.loc[
        df["primary_type"].str.contains(r"CRIM(?:INAL)? SEX(?:UAL)? ASSAULT", regex=True), 
        "primary_type"
    ] = "CRIMINAL SEXUAL ASSAULT"

    df.loc[
        df["primary_type"].str.contains("SEX OFFENSE"), 
        "primary_type"
    ] = "SEX OFFENSE"

    df.loc[
        df["primary_type"].str.contains("NARCOTIC"), 
        "primary_type"
    ] = "NARCOTICS"

    df.loc[
        df["primary_type"].str.contains("PUBLIC PEACE"), 
        "primary_type"
    ] = "PUBLIC PEACE VIOLATION"

    df.loc[
        df["primary_type"].str.contains("PUBLIC INDECENCY"), 
        "primary_type"
    ] = "PUBLIC INDECENCY"

    df.loc[
        df["primary_type"].str.contains("HOMICIDE"), 
        "primary_type"
    ] = "HOMICIDE"

    df.loc[
        df["primary_type"].str.contains("PROSTIT"), 
        "primary_type"
    ] = "PROSTITUTION"

    df.loc[
        df["primary_type"].str.contains("WEAPON"), 
        "primary_type"
    ] = "WEAPONS VIOLATION"

    df["primary_type"] = df["primary_type"].replace("", "UNKNOWN")
    df["primary_type"] = df["primary_type"].replace("NAN", "UNKNOWN")
    df["primary_type"] = df["primary_type"].fillna("UNKNOWN")

    return df

Proceeding with the location's description, this column was the most demanding on the preprocessing, as it consists of multiple different small categories. With that in mind. The ```clean_location_description(df)``` focus is not just on standardizing the format, but also on transforming and grouping the multiple small categories into more broad ones, making the dataset more readable

In [7]:
def clean_location_description(df):
    """
    Cleans and standardizes the 'location_description' column 
    in the Chicago crime dataset using pattern-based collapsing
    into simplified, analysis-friendly categories.
    """

    df["location_description"] = (
        df["location_description"]
        .astype(str)
        .str.upper()
        .str.strip()
    )

    df.loc[
        df["location_description"].str.contains(
            r"\bCHA\b|CHA GROUND|CHA ELEVATOR|CHA LOBBY|CHA STAIRWELL|CHA PARKING|CHA APARTMENT|CHA HALLWAY",
            regex=True
        ),
        "location_description"
    ] = "PUBLIC HOUSING"

    df.loc[
        df["location_description"].str.contains(
            r"RESIDENCE|HOUSE|APARTMENT|YARD|PORCH|HALLWAY|GARAGE|RESIDENTIAL|BASEMENT|ROOF|STAIRWELL|VESTIBULE|ABANDONED BUILDING|ELEVATOR"
        ),
        "location_description"
    ] = "RESIDENCE"

    df.loc[
        df["location_description"].str.contains(
            r"STREET|SIDEWALK|ALLEY|GANGWAY|DRIVEWAY|BRIDGE|HIGHWAY|EXPRESSWAY"
        ),
        "location_description"
    ] = "STREET / SIDEWALK"

    df.loc[
        df["location_description"].str.contains(
            r"PARKING|LOT|GARAGE"
        ),
        "location_description"
    ] = "PARKING AREA"

    df.loc[
        df["location_description"].str.contains(
            r"CTA|TRAIN|SUBWAY|BUS|L PLATFORM|L TRAIN|TRACKS|TRANSIT|RAILROAD"
        ),
        "location_description"
    ] = "PUBLIC TRANSIT"

    df.loc[
        df["location_description"].str.contains(
            r"STORE|RETAIL|GROCERY|MALL|GAS|DRUG STORE|PAWN|APPLIANCE|NEWSSTAND|DEALERSHIP|COIN OPERATED MACHINE"
        ),
        "location_description"
    ] = "RETAIL"

    df.loc[
        df["location_description"].str.contains(
            r"COMMERCIAL|BUSINESS|OFFICE|BANK|CREDIT UNION|SAVINGS AND LOAN|CAR WASH|CURRENCY EXCHANGE|ATM|KENNEL"
        ),
        "location_description"
    ] = "BUSINESS"

    df.loc[
        df["location_description"].str.contains(
            r"SCHOOL|UNIVERSITY|COLLEGE|DAY CARE|LIBRARY"
        ),
        "location_description"
    ] = "SCHOOL"

    df.loc[
        df["location_description"].str.contains(
            r"HOSPITAL|MEDICAL|DENTAL|ANIMAL HOSPITAL"
        ),
        "location_description"
    ] = "HOSPITAL"

    df.loc[
        df["location_description"].str.contains(
            r"AIRPORT|AIRCRAFT|ATS"
        ),
        "location_description"
    ] = "AIRPORT"

    df.loc[
        df["location_description"].str.contains(
            r"POLICE|GOVERNMENT|JAIL|LOCK-UP|COURT|FIRE STATION|FEDERAL BUILDING"
        ),
        "location_description"
    ] = "GOVERNMENT"

    df.loc[
        df["location_description"].str.contains(
            r"VEHICLE|TAXI|RIDE SHARE|DELIVERY TRUCK|TRUCK|AUTO"
        ),
        "location_description"
    ] = "VEHICLE"

    df.loc[
        df["location_description"].str.contains(
            r"CASINO|CLUB|TAVERN|LIQUOR|BANQUET HALL|POOL ROOM"
        ),
        "location_description"
    ] = "ENTERTAINMENT"

    df.loc[
        df["location_description"].str.contains(
            r"PARK|FOREST|BEACH|WATERFRONT|RIVER|LAKE|CEMETARY|FARM|BOAT"
        ),
        "location_description"
    ] = "PARK / OUTDOOR"

    df.loc[
        df["location_description"].str.contains(
            r"FACTORY|WAREHOUSE|MANUFACTURING"
        ),
        "location_description"
    ] = "INDUSTRIAL"

    df.loc[
        df["location_description"].str.contains(
            r"CHURCH|SYNAGOGUE|PLACE OF WORSHIP"
        ),
        "location_description"
    ] = "PLACE OF WORSHIP"

    # Handle missing / unknown
    df["location_description"] = df["location_description"].replace("NAN", "UNKNOWN")
    df["location_description"] = df["location_description"].fillna("UNKNOWN")

    return df


The arrest and domestic columns where boolean values, so just these functions are base on normalizing all possible values to make sure that the dataset is clean

In [8]:
def clean_arrest(df):
    df["arrest"] = df["arrest"].astype(str).str.lower().str.strip()

    arrest_map = {
        "true": True, "t": True, "1": True, "y": True, "yes": True,
        "false": False, "f": False, "0": False, "n": False, "no": False,
        "nan": False, "": False
    }

    df["arrest"] = df["arrest"].map(arrest_map).fillna(False)

    df["arrest"] = df["arrest"].astype("bool")

    return df

def clean_domestic(df):
    df["domestic"] = df["domestic"].astype(str).str.lower().str.strip()

    domestic_map = {
        "true": True, "t": True, "1": True, "y": True, "yes": True,
        "false": False, "f": False, "0": False, "n": False, "no": False,
        "nan": False, "": False
    }

    df["domestic"] = df["domestic"].map(domestic_map).fillna(False)
    df["domestic"] = df["domestic"].astype("bool")

    return df

After this the function were applied to the dataset, cleaning it, and final manually inspected

In [9]:
df = clean_primary_type(df)
df = clean_location_description(df)
df = clean_arrest(df)
df = clean_domestic(df)

df["date"] = pd.to_datetime(df["date"], errors="coerce")

df["date_only"] = df["date"].dt.date

crime_stats_clean = (
    df
    .groupby(["date_only", "primary_type","location_description", "arrest", "domestic"])
    .size()
    .reset_index(name="crime_count")
)

print(crime_stats_clean.head())

print(len(crime_stats_clean), "rows in grouped dataset\n\n")

print(df["primary_type"].unique(), "\n\n")

print(df["location_description"].unique(), "\n\n")

print(df["arrest"].unique())

print(df["domestic"].unique())

    date_only primary_type location_description  arrest  domestic  crime_count
0  2020-01-01      ASSAULT        ENTERTAINMENT   False      True            1
1  2020-01-01      ASSAULT        ENTERTAINMENT    True     False            1
2  2020-01-01      ASSAULT             HOSPITAL    True     False            1
3  2020-01-01      ASSAULT                OTHER   False     False            1
4  2020-01-01      ASSAULT                OTHER    True     False            1
305951 rows in grouped dataset


['BATTERY' 'THEFT' 'MOTOR VEHICLE THEFT' 'CRIMINAL DAMAGE' 'ASSAULT'
 'OFFENSE INVOLVING CHILDREN' 'OTHER OFFENSE' 'ROBBERY'
 'WEAPONS VIOLATION' 'INTERFERENCE WITH PUBLIC OFFICER' 'NARCOTICS'
 'SEX OFFENSE' 'PROSTITUTION' 'CRIMINAL TRESPASS' 'BURGLARY'
 'DECEPTIVE PRACTICE' 'PUBLIC PEACE VIOLATION' 'CRIMINAL SEXUAL ASSAULT'
 'LIQUOR LAW VIOLATION' 'ARSON' 'INTIMIDATION' 'STALKING' 'HOMICIDE'
 'CONCEALED CARRY LICENSE VIOLATION' 'HUMAN TRAFFICKING'
 'PUBLIC INDECENCY' 'KIDNAPPING' 'OBSCEN

In [14]:
crime_stats_clean.info(memory_usage="deep")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 305951 entries, 0 to 305950
Data columns (total 6 columns):
 #   Column                Non-Null Count   Dtype 
---  ------                --------------   ----- 
 0   date_only             305951 non-null  object
 1   primary_type          305951 non-null  object
 2   location_description  305951 non-null  object
 3   arrest                305951 non-null  bool  
 4   domestic              305951 non-null  bool  
 5   crime_count           305951 non-null  int64 
dtypes: bool(2), int64(1), object(3)
memory usage: 54.6 MB


To end the Data wragling process, the clean dataset is moved to the Supabase

In [ ]:
from sqlalchemy import create_engine, text
from dotenv import load_dotenv
import os

load_dotenv()

pg_url = (
    f"postgresql+psycopg2://{os.environ['PGUSER']}:{os.environ['PGPASSWORD']}"
    f"@{os.environ.get('PGHOST', 'localhost')}:{os.environ.get('PGPORT', '5432')}/"
    f"{os.environ['PGDATABASE']}"
)
engine = create_engine(pg_url, pool_pre_ping=True)

raw_df = df.copy()

with engine.begin() as conn:
    conn.execute(text("""
        CREATE TABLE IF NOT EXISTS clean_chicago_crimes (
            id BIGSERIAL PRIMARY KEY,
            date_only TEXT,
            primary_type TIMESTAMP,
            location_description TEXT,
            arrest TEXT,
            domestic TEXT,
            crime_count TEXT,
        )
    """))

raw_df.to_sql(
    "raw_chicago_crimes",
    engine,
    if_exists="append",
    index=False,
    method="multi",
    chunksize=5000,
)
print(f"Inserted {len(raw_df)} raw rows into Postgres")


Total records to insert: 251337
Inserted batch 0–5000, inserted rows: 5000
Inserted batch 5000–10000, inserted rows: 5000
Inserted batch 10000–15000, inserted rows: 5000
Inserted batch 15000–20000, inserted rows: 5000
Inserted batch 20000–25000, inserted rows: 5000
Inserted batch 25000–30000, inserted rows: 5000
Inserted batch 30000–35000, inserted rows: 5000
Inserted batch 35000–40000, inserted rows: 5000
Inserted batch 40000–45000, inserted rows: 5000
Inserted batch 45000–50000, inserted rows: 5000
Inserted batch 50000–55000, inserted rows: 5000
Inserted batch 55000–60000, inserted rows: 5000
Inserted batch 60000–65000, inserted rows: 5000
Inserted batch 65000–70000, inserted rows: 5000
Inserted batch 70000–75000, inserted rows: 5000
Inserted batch 75000–80000, inserted rows: 5000
Inserted batch 80000–85000, inserted rows: 5000
Inserted batch 85000–90000, inserted rows: 5000
Inserted batch 90000–95000, inserted rows: 5000
Inserted batch 95000–100000, inserted rows: 5000
Inserted batc